In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/nishantskuttan/mcq-solver/knowledge_base.faiss
/kaggle/input/datasets/nishantskuttan/mcq-solver/wandb_utils.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/rag_pipeline.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/knowledge_base_embeddings.npy
/kaggle/input/datasets/nishantskuttan/mcq-solver/retriever.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/reranker.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/faiss_index.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/submission.ipynb
/kaggle/input/datasets/nishantskuttan/mcq-solver/config.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/experiment.yaml
/kaggle/input/datasets/nishantskuttan/mcq-solver/embeddings.py
/kaggle/input/datasets/nishantskuttan/mcq-solver/__init__.py
/kaggle/input/data

In [2]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [3]:
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from tqdm.auto import tqdm

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# =============================================================================
# Approach 2 - RAG Model approach
# =============================================================================

In [5]:
# class KnowledgeBase:
#     """
#     Simple Knowledge Base used during inference.
#     """

#     def __init__(self):
#         self.documents = []

#     def build(self, train_df):
#         """
#         Extract the correct answer from every training question.
#         """

#         self.documents = [
#             str(row[row["answer"]])
#             for _, row in train_df.iterrows()
#         ]

#         return self.documents

#     def get_documents(self):
#         return self.documents

#     def __len__(self):
#         return len(self.documents)

In [6]:
# kb = KnowledgeBase()
# kb.build(train)

In [7]:
# from sentence_transformers import SentenceTransformer, CrossEncoder

# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [8]:
# knowledge_base = kb.get_documents()

In [9]:
# import numpy as np
# import faiss

# # knowledge_base_embeddings = np.load(
# #     "/kaggle/input/datasets/nishantskuttan/mcq-solver/knowledge_base_embeddings.npy"
# # )

# # faiss_index = faiss.read_index(
# #     "/kaggle/input/datasets/nishantskuttan/mcq-solver/knowledge_base.faiss"
# # )

# knowledge_base_embeddings = embedding_model.encode(
#     knowledge_base,
#     convert_to_numpy=True,
#     normalize_embeddings=True,
#     show_progress_bar=True,
# )

# faiss_index = faiss.IndexFlatIP(
#     knowledge_base_embeddings.shape[1]
# )

# faiss_index.add(knowledge_base_embeddings)

In [10]:
# class Retriever:

#     def __init__(self, embedding_model, faiss_index):
#         self.embedding_model = embedding_model
#         self.faiss_index = faiss_index

#     def retrieve(self, query, top_k):

#         query_embedding = self.embedding_model.encode(
#             [query],
#             normalize_embeddings=True,
#             convert_to_numpy=True,
#         )

#         distances, indices = self.faiss_index.search(
#             query_embedding,
#             top_k,
#         )

#         return distances[0], indices[0]

In [11]:
# retriever = Retriever(
#     embedding_model,
#     faiss_index,
# )

In [12]:
# print("FAISS dimension:", faiss_index.d)

In [13]:
# query_embedding = embedding_model.encode(
#     ["Hello World"],
#     normalize_embeddings=True,
#     convert_to_numpy=True,
# )

# print("Embedding shape:", query_embedding.shape)

In [14]:
# print("KB embeddings shape:", knowledge_base_embeddings.shape)

In [15]:
# query = test.iloc[0]["prompt"]

# distances, indices = retriever.retrieve(
#     query=query,
#     top_k=20,
# )

# print(indices)

In [16]:
class Reranker:

    def __init__(self, model):
        self.model = model

    def rerank(self, query, documents):

        pairs = [[query, document] for document in documents]

        scores = self.model.predict(pairs)

        ranked_indices = np.argsort(scores)[::-1]

        ranked_documents = [documents[i] for i in ranked_indices]
        ranked_scores = [scores[i] for i in ranked_indices]

        return ranked_documents, ranked_scores

In [17]:
# reranker = Reranker(reranker_model)

In [18]:
# query = test.iloc[0]["prompt"]

# _, indices = retriever.retrieve(
#     query=query,
#     top_k=10,
# )

# retrieved_documents = [
#     kb.get_documents()[index]
#     for index in indices
# ]

# ranked_documents, ranked_scores = reranker.rerank(
#     query=query,
#     documents=retrieved_documents,
# )

# for score, doc in zip(ranked_scores[:5], ranked_documents[:5]):
#     print(score)
#     print(doc)
#     print("-" * 80)

In [19]:
def retrieve_context(query, top_k=20):

    _, indices = retriever.retrieve(
        query=query,
        top_k=top_k,
    )

    retrieved_documents = [
        knowledge_base[i]
        for i in indices
    ]

    ranked_documents, ranked_scores = reranker.rerank(
        query=query,
        documents=retrieved_documents,
    )

    return ranked_documents, ranked_scores

In [20]:
def match_documents_to_options(
    ranked_documents,
    question_row,
    top_k=3,
):

    predictions = []
    seen = set()

    option_labels = ["A", "B", "C", "D", "E"]

    for document in ranked_documents:

        for label in option_labels:

            if document == str(question_row[label]):

                if label not in seen:
                    predictions.append(label)
                    seen.add(label)

                if len(predictions) == top_k:
                    return predictions

                break

    # Fallback
    for label in option_labels:

        if label not in seen:
            predictions.append(label)

        if len(predictions) == top_k:
            break

    return predictions

In [21]:
def predict(question_row, retrieve_top_k=20, predict_top_k=3):

    ranked_documents, _ = retrieve_context(
        query=str(question_row["prompt"]),
        top_k=retrieve_top_k,
    )

    predictions = match_documents_to_options(
        ranked_documents,
        question_row,
        top_k=predict_top_k,
    )

    return predictions

In [22]:
# prediction = predict(
#     test.iloc[0],
#     retrieve_top_k=20,
#     predict_top_k=3,
# )

# print(prediction)

In [23]:
# submission = pd.DataFrame({
#     "ID": test["id"],
#     "Prediction": [
#         " ".join(
#             predict(
#                 row,
#                 retrieve_top_k=20,
#                 predict_top_k=3,
#             )
#         )
#         for _, row in test.iterrows()
#     ]
# })

# submission.to_csv("/kaggle/working/submission.csv", index=False)

# submission.head()

# =============================================================================
# Approach 1 - Scoring with a base model
# =============================================================================

In [24]:
# import pandas as pd
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression

# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# OPTIONS = ["A","B","C","D","E"]

# rows = []

# for _, row in train.iterrows():
#     for opt in OPTIONS:
#         rows.append({
#             "text": f"{row['prompt']} [SEP] {row[opt]}",
#             "label": int(opt == row["answer"])
#         })

# pairwise = pd.DataFrame(rows)

# vec = TfidfVectorizer(
#     max_features=100000,
#     ngram_range=(1,3),
#     stop_words="english"
# )

# X = vec.fit_transform(pairwise["text"])

# model = LogisticRegression(
#     max_iter=3000,
#     class_weight="balanced",
#     n_jobs=-1
# )

# model.fit(X, pairwise["label"])

# preds = []

# for _, row in test.iterrows():

#     scores = {}

#     for opt in OPTIONS:

#         txt = f"{row['prompt']} [SEP] {row[opt]}"

#         score = model.predict_proba(
#             vec.transform([txt])
#         )[0,1]

#         scores[opt] = score

#     top3 = sorted(
#         scores,
#         key=scores.get,
#         reverse=True
#     )[:3]

#     preds.append(" ".join(top3))

# submission = pd.DataFrame({
#     "id": test["id"],
#     "Prediction": preds
# })

# submission.to_csv(
#     "submission.csv",
#     index=False
# )

In [25]:
def apk(actual, predicted, k=3):
    """
    Average Precision at K for one sample.
    """
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(df, predictor):
    scores = []

    for _, row in df.iterrows():
        pred = predictor(row)
        scores.append(apk(row["answer"], pred, k=3))

    return np.mean(scores)

In [26]:
# correct = 0

# for _, row in train.iterrows():
#     pred = predict(row)[0]  # top-1 prediction

#     if pred == row["answer"]:
#         correct += 1

# accuracy = correct / len(train)

# print("Training Top-1 Accuracy:", accuracy)

# ==============================================================================
# Approach 3 - CrossEncoder Pairwise Ranking
# ==============================================================================

In [27]:
# cross_encoder = reranker_model

In [28]:
# OPTIONS = ["A", "B", "C", "D", "E"]

# def predict_crossencoder(question_row):

#     prompt = str(question_row["prompt"])

#     pairs = [
#         [prompt, str(question_row[option])]
#         for option in OPTIONS
#     ]

#     scores = cross_encoder.predict(pairs)

#     ranking = sorted(
#         zip(OPTIONS, scores),
#         key=lambda x: x[1],
#         reverse=True,
#     )

#     return [label for label, _ in ranking[:3]]

In [29]:
# row = train.iloc[0]

# print("Correct :", row["answer"])
# print("Predicted:", predict_crossencoder(row))

In [30]:
# correct = 0

# for _, row in train.iterrows():

#     prediction = predict_crossencoder(row)[0]

#     if prediction == row["answer"]:
#         correct += 1

# accuracy = correct / len(train)

# print(f"CrossEncoder Top-1 Accuracy: {accuracy:.4f}")

In [31]:
# OPTIONS = ["A", "B", "C", "D", "E"]

# train_pairs = []

# for _, row in train.iterrows():

#     prompt = row["prompt"]
#     answer = row["answer"]

#     for option in OPTIONS:

#         train_pairs.append({
#             "text": prompt + " [SEP] " + str(row[option]),
#             "label": int(option == answer)
#         })

# train_pairs = pd.DataFrame(train_pairs)

# print(train_pairs.head())
# print(train_pairs.shape)

In [32]:
# train_embeddings = embedding_model.encode(
#     train_pairs["text"].tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

In [33]:
# from sklearn.linear_model import LogisticRegression

# clf = LogisticRegression(
#     max_iter=3000,
#     class_weight="balanced",
#     random_state=42
# )

# clf.fit(
#     train_embeddings,
#     train_pairs["label"]
# )

In [34]:
def predict_embedding_classifier(row):

    texts = [
        row["prompt"] + " [SEP] " + str(row[option])
        for option in OPTIONS
    ]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    probabilities = clf.predict_proba(embeddings)[:, 1]

    ranking = sorted(
        zip(OPTIONS, probabilities),
        key=lambda x: x[1],
        reverse=True
    )

    return [label for label, _ in ranking[:3]]

In [35]:
# correct = 0

# for _, row in train.iterrows():

#     prediction = predict_embedding_classifier(row)[0]

#     if prediction == row["answer"]:
#         correct += 1

# print(correct / len(train))

In [36]:
# score = mapk(train, predict_embedding_classifier)
# print(f"Training MAP@3: {score:.4f}")

# =============================================================================
# Approach 4 - PyTorch MLP (Built from Scratch)
# =============================================================================

In [37]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

from tqdm.auto import tqdm

In [38]:
OPTIONS = ["A","B","C","D","E"]

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [39]:
train_df, valid_df = train_test_split(

    train,

    test_size=0.20,

    random_state=42,

    stratify=train["answer"]

)

print(len(train_df))
print(len(valid_df))

1600
400


In [40]:
def create_pairs(df):

    rows = []

    for _, row in df.iterrows():

        prompt = row["prompt"]

        answer = row["answer"]

        for option in OPTIONS:

            rows.append({

                "text": prompt + " " + row[option],

                "label": int(option == answer)

            })

    return pd.DataFrame(rows)

In [41]:
train_pairs = create_pairs(train_df)

valid_pairs = create_pairs(valid_df)

print(train_pairs.shape)
print(valid_pairs.shape)

(8000, 2)
(2000, 2)


In [42]:
import re
from collections import Counter

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

counter = Counter()

for text in train_pairs["text"]:
    counter.update(tokenize(text))

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word, freq in counter.items():
    vocab[word] = len(vocab)

print(f"Vocabulary Size: {len(vocab)}")

Vocabulary Size: 2982


In [43]:
MAX_LEN = 192

def encode_text(text):

    tokens = tokenize(text)

    ids = [
        vocab.get(token, vocab[UNK_TOKEN])
        for token in tokens
    ]

    ids = ids[:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [vocab[PAD_TOKEN]] * (MAX_LEN - len(ids))

    return ids

In [44]:
vectorizer = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2)

)

X_train = vectorizer.fit_transform(
    train_pairs["text"]
)

X_valid = vectorizer.transform(
    valid_pairs["text"]
)

y_train = train_pairs["label"].values

y_valid = valid_pairs["label"].values

In [45]:
class MCQDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [46]:
X_train = np.array(
    [encode_text(text) for text in train_pairs["text"]],
    dtype=np.int64
)

X_valid = np.array(
    [encode_text(text) for text in valid_pairs["text"]],
    dtype=np.int64
)

y_train = train_pairs["label"].values
y_valid = valid_pairs["label"].values

print(X_train.shape)

(8000, 192)


In [47]:
train_loader = DataLoader(

    MCQDataset(X_train,y_train),

    batch_size=64,

    shuffle=True

)

valid_loader = DataLoader(

    MCQDataset(X_valid,y_valid),

    batch_size=64,

    shuffle=False

)

In [48]:
class MCQClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=256,
        lstm_hidden=256,
        hidden_dim=256,
        dropout=0.3
    ):
        super().__init__()

        # Word embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # Bi-LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

        # Attention layer
        self.attention = nn.Linear(lstm_hidden * 2, 1)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 1)
        )

    def forward(self, x):

        # x: (batch, seq_len)

        # Create mask for padding tokens
        mask = (x != 0)

        # Embedding
        embeddings = self.embedding(x)

        # Bi-LSTM outputs
        outputs, _ = self.lstm(embeddings)
        # outputs shape:
        # (batch, seq_len, lstm_hidden*2)

        # Attention scores
        attention_scores = self.attention(outputs).squeeze(-1)
        # (batch, seq_len)

        # Ignore padding positions
        attention_scores = attention_scores.masked_fill(
            ~mask,
            float('-inf')
        )

        # Attention weights
        attention_weights = F.softmax(
            attention_scores,
            dim=1
        )

        # Weighted sum of LSTM outputs
        context = torch.sum(
            outputs * attention_weights.unsqueeze(-1),
            dim=1
        )

        # Classification
        out = self.fc(context)

        return out

In [49]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MCQClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=256,
    dropout=0.3
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [50]:
def evaluate(model, loader, threshold=0.5):

    model.eval()

    losses = []
    preds = []
    labels = []

    with torch.no_grad():

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device).unsqueeze(1)

            logits = model(xb)

            loss = criterion(logits, yb)
            losses.append(loss.item())

            probs = torch.sigmoid(logits)
            pred = (probs > threshold).float()

            preds.extend(pred.cpu().numpy().flatten())
            labels.extend(yb.cpu().numpy().flatten())

    return (
        np.mean(losses),
        accuracy_score(labels, preds),
        f1_score(labels, preds)
    )

In [51]:
EPOCHS = 20

best_f1 = 0.0
patience = 5
counter = 0

history = {
    "train_loss": [],
    "valid_loss": [],
    "valid_acc": [],
    "valid_f1": []
}

for epoch in range(EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # -------------------------
    # Validation
    # -------------------------
    valid_loss, valid_acc, valid_f1 = evaluate(model, valid_loader)

    scheduler.step(valid_loss)

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)
    history["valid_f1"].append(valid_f1)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Acc: {valid_acc:.4f} | "
        f"F1: {valid_f1:.4f}"
    )

    # -------------------------
    # Early stopping
    # -------------------------
    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(model.state_dict(), "best_model.pt")

    else:

        counter += 1

        if counter >= patience:
            print("Early stopping.")
            break

Epoch 01 | Train Loss: 0.5067 | Valid Loss: 0.4436 | Acc: 0.8120 | F1: 0.1754
Epoch 02 | Train Loss: 0.3473 | Valid Loss: 0.2404 | Acc: 0.9055 | F1: 0.7192
Epoch 03 | Train Loss: 0.1870 | Valid Loss: 0.1460 | Acc: 0.9360 | F1: 0.8242
Epoch 04 | Train Loss: 0.1182 | Valid Loss: 0.0961 | Acc: 0.9530 | F1: 0.8669
Epoch 05 | Train Loss: 0.0930 | Valid Loss: 0.0863 | Acc: 0.9560 | F1: 0.8886
Epoch 06 | Train Loss: 0.0954 | Valid Loss: 0.1342 | Acc: 0.9430 | F1: 0.8343
Epoch 07 | Train Loss: 0.0772 | Valid Loss: 0.0459 | Acc: 0.9755 | F1: 0.9382
Epoch 08 | Train Loss: 0.0439 | Valid Loss: 0.0384 | Acc: 0.9780 | F1: 0.9420
Epoch 09 | Train Loss: 0.0372 | Valid Loss: 0.0450 | Acc: 0.9770 | F1: 0.9435
Epoch 10 | Train Loss: 0.0346 | Valid Loss: 0.0380 | Acc: 0.9815 | F1: 0.9544
Epoch 11 | Train Loss: 0.0214 | Valid Loss: 0.0191 | Acc: 0.9895 | F1: 0.9738
Epoch 12 | Train Loss: 0.0217 | Valid Loss: 0.0357 | Acc: 0.9835 | F1: 0.9581
Epoch 13 | Train Loss: 0.0311 | Valid Loss: 0.0131 | Acc: 0.9955

In [52]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [53]:
OPTIONS = ["A", "B", "C", "D", "E"]

test_sequences = []

for _, row in test.iterrows():

    option_sequences = []

    for option in OPTIONS:

        text = f"{row['prompt']} [SEP] {row[option]}"
        option_sequences.append(encode_text(text))

    test_sequences.append(option_sequences)

test_sequences = np.array(test_sequences, dtype=np.int64)

print(test_sequences.shape)

(500, 5, 192)


In [54]:
model.eval()

all_scores = []

with torch.no_grad():

    for option in range(5):

        X = torch.LongTensor(test_sequences[:, option, :]).to(device)

        logits = model(X).squeeze(1)

        scores = torch.sigmoid(logits).cpu().numpy()

        all_scores.append(scores)

all_scores = np.stack(all_scores, axis=1)

print(all_scores.shape)

(500, 5)


In [55]:
option_letters = np.array(["A", "B", "C", "D", "E"])

predictions = []

for scores in all_scores:

    ranked = option_letters[np.argsort(scores)[::-1]]

    predictions.append(" ".join(ranked[:3]))

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A C D
1,2,B C E
2,3,B C E
3,4,E C A
4,5,C A E
